# ElectInfo publishes TX-32 precincts via a GeoDjango API

## What this shows
How ElectInfo wraps the unified precinct GeoDataFrame from `spatial/05` into a GeoDjango model backed by PostGIS, exposes a single `FeatureCollection` JSON endpoint, and stages a docker-compose snippet for local PostGIS so the pattern is concretely copy-pasteable.

## Why it matters
Dashboards and external integrations need a read-only HTTP surface — not a notebook. GeoDjango keeps the ORM / admin / serializer path first-class, which is what takes a one-off analysis notebook to a production-grade service.

## Prereqs
- `pip install 'siege-utilities[geo]'` + `Django` + `psycopg2-binary` (for PostGIS) or just SQLite-SpatiaLite for local dev.
- PostGIS for production; SpatiaLite fallback for laptop dev.

## Next
- `spatial/05_multi_source_joins.ipynb` — the precinct data this service exposes.
- `engines/03_databricks_geo.ipynb` — an alternative production surface for geo data.


## 1. PostGIS via docker-compose (dev)

This is the canonical local setup — one container, named volume, the standard `postgis/postgis` image. ElectInfo developers run this on their laptop; production uses RDS / Cloud SQL with the PostGIS extension enabled the same way.

```yaml
# docker-compose.yml
services:
  postgis:
    image: postgis/postgis:16-3.4
    environment:
      POSTGRES_USER: electinfo
      POSTGRES_PASSWORD: devsecret
      POSTGRES_DB: electinfo_dev
    ports: ['5432:5432']
    volumes:
      - postgis_data:/var/lib/postgresql/data
volumes:
  postgis_data:
```

```bash
docker compose up -d
```

## 2. GeoDjango model for TX-32 precincts (call shape)

```python
# app/models.py
from django.contrib.gis.db import models

class Precinct(models.Model):
    precinct_id  = models.CharField(max_length=32, primary_key=True)
    name_display = models.CharField(max_length=128)
    dem_pct      = models.FloatField(null=True)
    turnout_pct  = models.FloatField(null=True)
    quality_flag = models.CharField(max_length=16, default='high')
    geom         = models.PolygonField(srid=4326)

    class Meta:
        indexes = [
            models.Index(fields=['quality_flag']),
            models.Index(fields=['dem_pct']),
        ]
```

## 3. Load the unified DataFrame from `spatial/05` as fixtures (call shape)

ElectInfo's loader is a small management command. It takes the joined DataFrame, converts each row into a `Precinct` instance, and bulk-creates. Idempotent via `update_or_create`, so the weekly refresh job can rerun without duplicating.

```python
# app/management/commands/load_precincts.py
from django.core.management.base import BaseCommand
from django.contrib.gis.geos import GEOSGeometry
from app.models import Precinct

class Command(BaseCommand):
    def handle(self, *args, **opts):
        from spatial_05_unified import build_unified_dataframe  # pipeline fn
        gdf = build_unified_dataframe()
        for _, row in gdf.iterrows():
            Precinct.objects.update_or_create(
                precinct_id=row['precinct_id'],
                defaults={
                    'name_display': row['name_display'],
                    'dem_pct':      row['dem_pct'],
                    'turnout_pct':  row['turnout_pct'],
                    'quality_flag': row['quality_flag'],
                    'geom':         GEOSGeometry(row['geometry'].wkt, srid=4326),
                },
            )
```

## 4. A single GeoJSON FeatureCollection endpoint (call shape)

Django's `serializers.serialize('geojson', ...)` is the production call. One view, one URL.

```python
# app/views.py
from django.core.serializers import serialize
from django.http import HttpResponse
from app.models import Precinct

def precincts_geojson(request):
    qs = Precinct.objects.filter(quality_flag='high')
    data = serialize(
        'geojson', qs,
        geometry_field='geom',
        fields=('precinct_id', 'name_display', 'dem_pct', 'turnout_pct'),
    )
    return HttpResponse(data, content_type='application/json')

# app/urls.py
from django.urls import path
from app.views import precincts_geojson

urlpatterns = [path('precincts.geojson', precincts_geojson)]
```

```bash
curl -s http://localhost:8000/precincts.geojson | jq '.features[0]'
```

## 5. Why this notebook doesn't execute the Django server inline

Running a Django process inside nbconvert means spinning up the ORM, connecting to PostGIS, running migrations — all of which depend on a live DB the notebook can't safely provision. The cells above are a complete, copy-pasteable recipe; execute them in a proper Django project (or the `siege_utilities/geo/geodjango_*` helpers) with a live PostGIS instance.

## Related

- **Source**: `siege_utilities/geo/geodjango_*` for the GeoDjango integration helpers.
- **Tests**: `tests/test_geodjango*.py` where applicable.
- **Next**: `engines/03_databricks_geo.ipynb` for the cloud-native alternative; `spatial/05_multi_source_joins.ipynb` for the data that feeds this service.
